# NSE F&O Data — Exploration & Loading

**Dataset:** NSE Future and Options Dataset 3M  
**Source:** https://www.kaggle.com/datasets/sunnysai12345/nse-future-and-options-dataset-3m  
**Database:** fno_db (PostgreSQL)

This notebook loads the NSE F&O dataset into PostgreSQL after basic exploration and cleaning.

In [1]:
import pandas as pd
import psycopg2
from psycopg2.extras import execute_values

## 1. Load Dataset

In [2]:
df = pd.read_csv('../data/3mfanddo.csv')
df.head()

,Unnamed: 0,INSTRUMENT,SYMBOL,EXPIRY_DT,STRIKE_PR,OPTION_TYP,OPEN,HIGH,LOW,CLOSE,SETTLE_PR,CONTRACTS,VAL_INLAKH,OPEN_INT,CHG_IN_OI,TIMESTAMP
0,160393,FUTIDX,BANKNIFTY,29-Aug-2019,0.0,XX,28805.65,28924.00,28140.55,28499.30,28499.30,214569.0,1225914.96,1675780.0,234640.0,01-AUG-2019
1,160394,FUTIDX,BANKNIFTY,26-Sep-2019,0.0,XX,28926.40,29030.55,28251.70,28611.45,28611.45,2484.0,14245.95,51400.0,-80.0,01-AUG-2019
2,160395,FUTIDX,BANKNIFTY,31-Oct-2019,0.0,XX,29000.00,29105.00,28355.55,28699.05,28699.05,598.0,3434.43,9460.0,4860.0,01-AUG-2019
3,160396,FUTIDX,NIFTY,29-Aug-2019,0.0,XX,11098.40,11098.40,10901.10,11015.35,11015.35,199881.0,1650955.24,19001400.0,1339200.0,01-AUG-2019
4,160397,FUTIDX,NIFTY,26-Sep-2019,0.0,XX,11136.35,11145.20,10955.00,11066.60,11066.60,5283.0,43841.57,893625.0,66750.0,01-AUG-2019


In [ ]:

# drop unnamed index column
df.drop(columns=['Unnamed: 0'], inplace=True, errors='ignore')
# lowercase column names for better work with SQL
df.columns = [c.lower() for c in df.columns]

print('Shape:', df.shape)
df.head(3)

Shape: (2533210, 15)


,instrument,symbol,expiry_dt,strike_pr,option_typ,open,high,low,close,settle_pr,contracts,val_inlakh,open_int,chg_in_oi,timestamp
0,FUTIDX,BANKNIFTY,29-Aug-2019,0.0,XX,28805.65,28924.00,28140.55,28499.30,28499.30,214569.0,1225914.96,1675780.0,234640.0,01-AUG-2019
1,FUTIDX,BANKNIFTY,26-Sep-2019,0.0,XX,28926.40,29030.55,28251.70,28611.45,28611.45,2484.0,14245.95,51400.0,-80.0,01-AUG-2019
2,FUTIDX,BANKNIFTY,31-Oct-2019,0.0,XX,29000.00,29105.00,28355.55,28699.05,28699.05,598.0,3434.43,9460.0,4860.0,01-AUG-2019


## 2. Basic Exploration

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2533210 entries, 0 to 2533209
Data columns (total 15 columns):
 #   Column      Dtype  
---  ------      -----  
 0   instrument  object 
 1   symbol      object 
 2   expiry_dt   object 
 3   strike_pr   float64
 4   option_typ  object 
 5   open        float64
 6   high        float64
 7   low         float64
 8   close       float64
 9   settle_pr   float64
 10  contracts   float64
 11  val_inlakh  float64
 12  open_int    float64
 13  chg_in_oi   float64
 14  timestamp   object 
dtypes: float64(10), object(5)
memory usage: 289.9+ MB


In [5]:
df.describe()

,strike_pr,open,high,low,close,settle_pr,contracts,val_inlakh,open_int,chg_in_oi
count,2.533210e+06,2.533210e+06,2.533210e+06,2.533210e+06,2.533210e+06,2.533210e+06,2.533210e+06,2.533210e+06,2.533210e+06,2.533210e+06
mean,3.681784e+03,3.458899e+01,3.630275e+01,3.303485e+01,3.076770e+02,3.210406e+02,6.242578e+02,4.061064e+03,1.573582e+05,2.429049e+03
std,7.834708e+03,6.576684e+02,6.661756e+02,6.502584e+02,1.064385e+03,1.140909e+03,2.070733e+04,1.293172e+05,3.488634e+06,4.554198e+05
min,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,5.000000e-02,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,-1.328880e+08
25%,2.000000e+02,0.000000e+00,0.000000e+00,0.000000e+00,7.100000e+00,3.350000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
50%,5.800000e+02,0.000000e+00,0.000000e+00,0.000000e+00,3.620000e+01,2.920000e+01,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
75%,1.900000e+03,0.000000e+00,0.000000e+00,0.000000e+00,1.632500e+02,1.477000e+02,0.000000e+00,0.000000e+00,8.000000e+02,0.000000e+00
max,7.100000e+04,6.664885e+04,6.752930e+04,6.629995e+04,6.664530e+04,6.691670e+04,4.564524e+06,2.875726e+07,6.323800e+08,1.383200e+08


In [6]:
df.isnull().sum()

instrument    0
symbol        0
expiry_dt     0
strike_pr     0
option_typ    0
open          0
high          0
low           0
close         0
settle_pr     0
contracts     0
val_inlakh    0
open_int      0
chg_in_oi     0
timestamp     0
dtype: int64

In [ ]:
print('INSTRUMENT types :', df['instrument'].unique())
print('OPTION_TYP values:', df['option_typ'].unique())
print('Total symbols    :', df['symbol'].nunique())
print('Expiry dates     :', df['expiry_dt'].nunique())

print('Date range (timestamp):', df['timestamp'].min(), 'to', df['timestamp'].max())

INSTRUMENT types : ['FUTIDX' 'FUTSTK' 'OPTIDX' 'OPTSTK']
OPTION_TYP values: ['XX' 'CE' 'PE']
Total symbols    : 164
Expiry dates     : 37
Date range (timestamp): 01-AUG-2019 to 31-OCT-2019


**Observations:**
- `instrument` has 3 types: `FUTIDX` (index futures), `FUTSTK` (stock futures), `OPTIDX` (index options)
- `option_typ = 'XX'` for all futures rows — sentinel value in the source, not a null
- `strike_pr = 0` for all futures — futures have no strike price
- `settle_pr = 0` for some options — settlement price only populated at contract expiry

In [ ]:
# num descriptions- checking price and volume ranges
df[['open','high','low','close','contracts','open_int','chg_in_oi']].describe().round(2)

,open,high,low,close,contracts,open_int,chg_in_oi
count,2533210.00,2533210.00,2533210.00,2533210.00,2533210.00,2.533210e+06,2.533210e+06
mean,34.59,36.30,33.03,307.68,624.26,1.573582e+05,2.429050e+03
std,657.67,666.18,650.26,1064.39,20707.33,3.488634e+06,4.554198e+05
min,0.00,0.00,0.00,0.05,0.00,0.000000e+00,-1.328880e+08
25%,0.00,0.00,0.00,7.10,0.00,0.000000e+00,0.000000e+00
50%,0.00,0.00,0.00,36.20,0.00,0.000000e+00,0.000000e+00
75%,0.00,0.00,0.00,163.25,0.00,8.000000e+02,0.000000e+00
max,66648.85,67529.30,66299.95,66645.30,4564524.00,6.323800e+08,1.383200e+08


## 3. Clean & Prepare

In [ ]:
# dates are in DD-Mon-YYYY format ie. '01-AUG-2019' 
df['timestamp'] = pd.to_datetime(df['timestamp'], format='%d-%b-%Y')
df['expiry_dt'] = pd.to_datetime(df['expiry_dt'], format='%d-%b-%Y')

df['strike_pr'] = df['strike_pr'].fillna(0)
df['option_typ'] = df['option_typ'].fillna('XX')

print('Date range:', df['timestamp'].min().date(), 'to', df['timestamp'].max().date())

Date range: 2019-08-01 to 2019-11-15


In [10]:
df.loc[:, ['timestamp', 'expiry_dt']].head()

,timestamp,expiry_dt
0,2019-08-01,2019-08-29
1,2019-08-01,2019-09-26
2,2019-08-01,2019-10-31
3,2019-08-01,2019-08-29
4,2019-08-01,2019-09-26


## 4. Load into PostgreSQL

Loading order: `exchanges` (already loaded) → `instruments` → `expiries` → `trades`  
due to FK constraints require this sequence.

In [ ]:
conn = psycopg2.connect(
    host='localhost',
    database='fno_db',
    user='postgres',
    password='postgres123'
)
cur = conn.cursor()
print('Connected to fno_db.')

Connected to fno_db.


In [ ]:
# all rows in this dataset belong to NSE,hence
cur.execute("SELECT exchange_id FROM exchanges WHERE exchange_name = 'NSE'")
nse_id = cur.fetchone()[0]

instruments_df = df[['symbol', 'instrument']].drop_duplicates().rename(columns={'instrument': 'instrument_type'})
instruments_df['exchange_id'] = nse_id

data = list(instruments_df[['exchange_id', 'symbol', 'instrument_type']].itertuples(index=False, name=None))

#Inserting values in instruments table
execute_values(cur,
    "INSERT INTO instruments (exchange_id, symbol, instrument_type) VALUES %s ON CONFLICT (exchange_id, symbol, instrument_type) DO NOTHING",
    data
)

conn.commit()
print('Instruments inserted:', len(data))

Instruments inserted: 328


In [ ]:
expiries_df = df[['expiry_dt', 'strike_pr', 'option_typ']].drop_duplicates()
data = list(expiries_df.itertuples(index=False, name=None))

#Inserting values in expiries table
execute_values(cur,
    "INSERT INTO expiries (expiry_dt, strike_pr, option_typ) VALUES %s ON CONFLICT (expiry_dt, strike_pr, option_typ) DO NOTHING",data
)
conn.commit()
print('Expiries inserted:', len(data))

Expiries inserted: 18232


In [ ]:
# fetch IDs from DB to map FK columns in trades table
cur.execute('SELECT instrument_id, symbol, instrument_type FROM instruments')
inst_map = {(row[1], row[2]): row[0] for row in cur.fetchall()}

cur.execute('SELECT expiry_id, expiry_dt, strike_pr, option_typ FROM expiries')
exp_map = {(row[1], float(row[2]), row[3]): row[0] for row in cur.fetchall()}

print('Instrument map size:', len(inst_map))

print('Expiry map size    :', len(exp_map))

Instrument map size: 328
Expiry map size    : 18232


In [ ]:
df['instrument_id'] = df.apply(lambda r: inst_map.get((r['symbol'], r['instrument'])), axis=1)
df['expiry_id']     = df.apply(lambda r: exp_map.get((r['expiry_dt'].date(), float(r['strike_pr']), r['option_typ'])), axis=1)
df['trade_date']    = df['timestamp'].dt.date

#rechecking the missing rows if any
missing = df['instrument_id'].isna().sum() + df['expiry_id'].isna().sum()
print('Rows with missing FK mapping:', missing)
df.dropna(subset=['instrument_id', 'expiry_id'], inplace=True)

Rows with missing FK mapping: 0


In [ ]:
#Inserting values in trades  table

trade_cols = [
    'instrument_id', 'expiry_id', 'trade_date',
    'open', 'high', 'low', 'close', 'settle_pr',
    'contracts', 'val_inlakh', 'open_int', 'chg_in_oi'
]

data = list(df[trade_cols].itertuples(index=False, name=None))

execute_values(cur,
    """
    INSERT INTO trades (
        instrument_id, expiry_id, trade_date,
        open, high, low, close, settle_pr,
        contracts, val_inlakh, open_int, chg_in_oi
    ) VALUES %s
    """,
    data,
    page_size=10000
)
conn.commit()
print('Trades inserted:', len(data))

Trades inserted: 2533210


In [17]:
# sanity check
for table in ['exchanges', 'instruments', 'expiries', 'trades']:
    cur.execute(f'SELECT COUNT(*) FROM {table}')
    print(f'{table:15s}: {cur.fetchone()[0]:,} rows')

cur.close()
conn.close()
print('\nConnection closed.')

exchanges      : 3 rows
instruments    : 328 rows
expiries       : 18,232 rows
trades         : 2,533,210 rows

Connection closed.
